# Rwanda VACS 2015–16 — merged dataset exploration

Primary extract **`Rwanda VACYS 2015-16 Final Data set.dta`** (filename spelling **VACYS**) in **`data/raw/Rwanda Stata and SAS/`**. **`pyreadstat.read_dta`** reads it with **default** encoding here (retry **`latin1`** if needed).

**Companion materials:** **`VACS_analysis_dofile.do`** documents CDC-style survey setup, e.g. **`svyset psu_id [pweight = child_wght], strata(prov)`**. SAS programs (**`Rwanda PV_040517.sas`**, **`Rwanda SV_040517.sas`**, **`Rwanda overall_040517.sas`**) support labeling / PV–SV splits; **ID/geo/weight** mapping here is **from the `.dta` + do-file**.

**About `id_dhs`:** Stata label **EA ID** (~250 EAs here)—**not** one respondent per row; many adolescents share an EA. **§1** shows only **raw** columns from the `.dta`. **§2b** adds composite row keys (`psu_id`+`hh`, `id_dhs`+`hh`) for later EDA.

**Flow:** §1 Load (raw ID/design) → §2 column list & EDA → **checklist mapping** (candidate columns) → **§2b** derived row keys → §3 samples & slot summaries → §4 harmonized TSV.


In [1]:
from pathlib import Path
import sys

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COUNTRY_DIR = ROOT / "data" / "raw" / "Rwanda Stata and SAS"
PUD_PATH = COUNTRY_DIR / "Rwanda VACYS 2015-16 Final Data set.dta"
DO_PATH = COUNTRY_DIR / "VACS_analysis_dofile.do"
READ_KW = {}

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")



## 1. Load data

**Raw extract only:** preview table is limited to **identifying / design** columns from the file (no concatenated IDs).


In [2]:
if not PUD_PATH.is_file():
    raise FileNotFoundError(PUD_PATH)

df, meta = pyreadstat.read_dta(PUD_PATH, **READ_KW)
print(f"File: {PUD_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")
if DO_PATH.is_file():
    print(f"Do-file (reference): {DO_PATH.name}")

df = df.copy()

if "sex" in df.columns:
    print("sex (confirm 1=male / 2=female in documentation):")
    display(df["sex"].value_counts(dropna=False).sort_index())

# --- Raw identifying / design columns only (no derived keys in §1) ---
print(
    f"id_dhs unique (EA level): {df['id_dhs'].nunique():,} — not one row per respondent; "
    f"total rows: {len(df):,}"
)
print(
    f"Rows per id_dhs — min {int(df.groupby('id_dhs').size().min())}, "
    f"median {df.groupby('id_dhs').size().median():.1f}, "
    f"max {int(df.groupby('id_dhs').size().max())}"
)
print(f"Max rows per psu_id: {int(df.groupby('psu_id').size().max())}")
print(f"Duplicate rows (all columns): {int(df.duplicated().sum())}")

_h = df["hdate_vf"].dropna()
if len(_h):
    print(
        "hdate_vf (numeric YYYYMMDD) min/max:",
        int(_h.astype(np.int64).min()),
        int(_h.astype(np.int64).max()),
    )

_raw_preview = [
    "id_dhs", "psu_id", "hh", "sex", "prov", "dist", "sect", "cel", "vill",
    "area", "int_cod", "hcluster", "visit_nf", "hdate_vf", "child_wght", "hh_wght",
]
assert all(c in df.columns for c in _raw_preview), _raw_preview
df[_raw_preview].head(4)


File: /Users/starsrain/research_side_projects_ipv/data/raw/Rwanda Stata and SAS/Rwanda VACYS 2015-16 Final Data set.dta
Rows × columns: 2,212 × 1,714
Do-file (reference): VACS_analysis_dofile.do
sex (confirm 1=male / 2=female in documentation):


sex
1.0    1180
2.0    1032
Name: count, dtype: int64

id_dhs unique (EA level): 250 — not one row per respondent; total rows: 2,212
Rows per id_dhs — min 2, median 9.0, max 18
Max rows per psu_id: 18
Duplicate rows (all columns): 0
hdate_vf (numeric YYYYMMDD) min/max: 20151101 20151223


,id_dhs,psu_id,hh,sex,prov,dist,sect,cel,vill,area,int_cod,hcluster,visit_nf,hdate_vf,child_wght,hh_wght
0,1.0,"1,102,030,101",78.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1011.0,1.0,1.0,20151221.0,1277.647095,2376.181152
1,1.0,"1,102,030,101",86.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1012.0,1.0,1.0,20151221.0,1277.647095,2376.181152
2,1.0,"1,102,030,101",26.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1013.0,1.0,1.0,20151222.0,1277.647095,2376.181152
3,1.0,"1,102,030,101",172.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1013.0,1.0,1.0,20151222.0,1277.647095,2376.181152


## 2. Column list & quick EDA

Uses **`df` straight from §1** (only columns in the `.dta`). After **§2b**, two derived columns are appended (`RESP_KEY`, `RESP_KEY_EAHH`).


In [3]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}

var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})
print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
display(var_table.head(40))
display(var_table.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))
df.info(max_cols=18)


Variables: 1,714  |  Observations: 2,212


,column,stata_label,dtype,missing_n,missing_pct
id_dhs,id_dhs,EA ID,float64,0,0.00
hh,hh,HOUSEHOLD,float64,0,0.00
prov,prov,PROVINCE,float64,0,0.00
dist,dist,DISTRICT,float64,0,0.00
sect,sect,SECT,float64,0,0.00
cel,cel,CELLULE,float64,0,0.00
vill,vill,VILLAGE,float64,0,0.00
area,area,AREA,float64,2212,100.00
int_cod,int_cod,INTERVIEWER CODE,float64,0,0.00
hcluster,hcluster,CLUSTER TYPE,float64,0,0.00


,column,stata_label,dtype,missing_n,missing_pct
0,ermth_22,DATE OF BIRTH OF ER -MONTH,float64,2212,100.0
1,er_19,Eligible Household Member,float64,2212,100.0
2,er_17,Eligible Household Member,float64,2212,100.0
3,er_16,Eligible Household Member,float64,2212,100.0
4,er_15,Eligible Household Member,float64,2212,100.0
5,er_14,Eligible Household Member,float64,2212,100.0
6,ermth_25,DATE OF BIRTH OF ER -MONTH,float64,2212,100.0
7,ermth_26,DATE OF BIRTH OF ER -MONTH,float64,2212,100.0
8,ermth_27,DATE OF BIRTH OF ER -MONTH,float64,2212,100.0
9,ermth_28,DATE OF BIRTH OF ER -MONTH,float64,2212,100.0


<class 'pandas.DataFrame'>
RangeIndex: 2212 entries, 0 to 2211
Columns: 1714 entries, id_dhs to age1stmean1317
dtypes: float64(1571), str(143)
memory usage: 28.9 MB


### Harmonized geography / ID checklist (you map columns yourself)

**Admin 2 = enumeration areas (EAs).** Sector / cell / village are **finer** geography (optional third geo level), not substitutes for EA when EA columns exist.

Match the extract to your template (**admin 1** → **district** if needed → **Admin 2 / EAs** → finer geo optional → household → person → field date → sex). After §2, **you** choose which columns belong in **`CANDIDATES`**; **`utils/checklist.py`** only summarizes columns that exist in `df` (plus stub rows when a slot is empty or a name is missing).

The next cell builds **`checklist_df`** via **`build_checklist_df`** and prints **TSV** for Excel. **`type_and_width`** combines short dtype + PI width (e.g. **`float; 1 digit`**, **`str; 8 digits`**) and, when values match a calendar pattern, a **layout** token (e.g. numeric **`YYYYMMDD`**, pandas datetime **`YYYY-MM-DD`** or **`YYYY-MM-DD HH:MM:SS`**). **`suggested_layout`** repeats that token alone for quick scanning. Helpers live under **`utils/`**.

**PI “digits”:** same as **character count** of the usual printed value (no zero-padding): codes **1–5** → **1** character; **15** → **2**; string IDs use **string length** as stored.


In [4]:
# You choose candidate columns after §2 EDA; utils only summarize what exists in `df`.
from utils.checklist import build_checklist_df, checklist_to_tsv

CANDIDATES = [
    ("Admin 1 (province)", ["prov"]),
    ("Admin ~1.5 (district)", ["dist"]),
    ("Admin 2 — enumeration area (EA), priority", ["id_dhs", "ea_code", "psu_id"]),
    ("Finer admin / optional (sector, cell, village)", ["sect", "cel", "vill"]),
    ("Cluster type (design)", ["hcluster"]),
    ("Household # (within PSU)", ["hh"]),
    ("Individual ID (if any)", []),
    ("Field / visit date", ["hdate_vf", "visit_nf"]),
    ("Sex", ["sex"]),
]

_labels = meta.column_names_to_labels or {}
checklist_df = build_checklist_df(df, CANDIDATES, column_labels=_labels)

with pd.option_context("display.max_colwidth", 100, "display.width", 220):
    display(checklist_df)

print("\n--- TSV (copy for Excel / codebook) ---\n")
print(checklist_to_tsv(checklist_df))



,slot,column,stata_label,type_and_width,suggested_layout,dtype,nunique,missing_n,missing_pct,pi_digits_char_usual_display,min_nonnull,max_nonnull,sample_first_3,slot_notes
0,Admin 1 (province),prov,PROVINCE,float; 1 digit,NaN,float64,5,0,0.0,1,1.0,5.0,"1.0, 1.0, 1.0",<NA>
1,Admin ~1.5 (district),dist,DISTRICT,float; 1 digit,NaN,float64,8,0,0.0,1,1.0,8.0,"1.0, 1.0, 1.0",<NA>
2,"Admin 2 — enumeration area (EA), priority",id_dhs,EA ID,float; 1–3 digits,NaN,float64,250,0,0.0,1–3,1.0,250.0,"1.0, 1.0, 1.0",<NA>
3,"Admin 2 — enumeration area (EA), priority",ea_code,EA_code,str; 8 digits,NaN,str,250,0,0.0,8,<NA>,<NA>,"'11020301', '11020301', '11020301'",<NA>
4,"Admin 2 — enumeration area (EA), priority",psu_id,Primary sampling unit ID,str; 13 digits,NaN,str,250,0,0.0,13,<NA>,<NA>,"'1,102,030,101', '1,102,030,101', '1,102,030,101'",<NA>
5,"Finer admin / optional (sector, cell, village)",sect,SECT,float; 1–2 digits,NaN,float64,18,0,0.0,1–2,1.0,18.0,"2.0, 2.0, 2.0",<NA>
6,"Finer admin / optional (sector, cell, village)",cel,CELLULE,float; 1 digit,NaN,float64,9,0,0.0,1,1.0,9.0,"3.0, 3.0, 3.0",<NA>
7,"Finer admin / optional (sector, cell, village)",vill,VILLAGE,float; 1–2 digits,NaN,float64,14,0,0.0,1–2,1.0,15.0,"1.0, 1.0, 1.0",<NA>
8,Cluster type (design),hcluster,CLUSTER TYPE,float; 1 digit,NaN,float64,2,0,0.0,1,1.0,2.0,"1.0, 1.0, 1.0",<NA>
9,Household # (within PSU),hh,HOUSEHOLD,float; 1–3 digits,NaN,float64,218,0,0.0,1–3,1.0,249.0,"78.0, 86.0, 26.0",<NA>



--- TSV (copy for Excel / codebook) ---

slot	column	stata_label	type_and_width	suggested_layout	dtype	nunique	missing_n	missing_pct	pi_digits_char_usual_display	min_nonnull	max_nonnull	sample_first_3	slot_notes
Admin 1 (province)	prov	PROVINCE	float; 1 digit		float64	5	0	0.0	1	1.0	5.0	1.0, 1.0, 1.0	
Admin ~1.5 (district)	dist	DISTRICT	float; 1 digit		float64	8	0	0.0	1	1.0	8.0	1.0, 1.0, 1.0	
Admin 2 — enumeration area (EA), priority	id_dhs	EA ID	float; 1–3 digits		float64	250	0	0.0	1–3	1.0	250.0	1.0, 1.0, 1.0	
Admin 2 — enumeration area (EA), priority	ea_code	EA_code	str; 8 digits		str	250	0	0.0	8			'11020301', '11020301', '11020301'	
Admin 2 — enumeration area (EA), priority	psu_id	Primary sampling unit ID	str; 13 digits		str	250	0	0.0	13			'1,102,030,101', '1,102,030,101', '1,102,030,101'	
Finer admin / optional (sector, cell, village)	sect	SECT	float; 1–2 digits		float64	18	0	0.0	1–2	1.0	18.0	2.0, 2.0, 2.0	
Finer admin / optional (sector, cell, village)	cel	CELLULE	float; 1 digit		

## 2b. Derived row keys (for EDA below)

These composites are **not** in the Stata file; they are built for **unique row tracking** and slot summaries. **`id_dhs`** stays the raw EA identifier.


In [ ]:
df["RESP_KEY"] = df["psu_id"].astype(str) + "_" + df["hh"].astype(int).astype(str)
df["RESP_KEY_EAHH"] = (
    df["id_dhs"].astype(int).astype(str) + "_" + df["hh"].astype(int).astype(str)
)
assert df["RESP_KEY"].nunique() == len(df), "RESP_KEY not unique—update construction"
assert df["RESP_KEY_EAHH"].nunique() == len(df), "RESP_KEY_EAHH not unique—update construction"
assert int(df.groupby("psu_id")["id_dhs"].nunique().max()) == 1
assert int(df.groupby("id_dhs")["psu_id"].nunique().max()) == 1
print("Derived RESP_KEY, RESP_KEY_EAHH — both unique; psu_id ↔ id_dhs one-to-one in this extract.")


## 3. Further EDA and exploration

Run **§2b** first so **`RESP_KEY`** / **`RESP_KEY_EAHH`** exist.

### Raw row samples


In [11]:
pd.set_option("display.max_columns", 42)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 72)

_core = [c for c in [
    "RESP_KEY", "RESP_KEY_EAHH", "id_dhs", "psu_id", "hh", "sex", "prov", "dist", "sect",
    "cel", "vill", "area", "hcluster", "child_wght", "hh_wght",
    "hdate_vf", "q1_hh", "q1_mm",
] if c in df.columns]
display(df[_core].head(8))
display(df[_core].sample(6, random_state=0))


,RESP_KEY,RESP_KEY_EAHH,id_dhs,psu_id,hh,sex,prov,dist,sect,cel,vill,area,hcluster,child_wght,hh_wght,hdate_vf,q1_hh,q1_mm
0,"1,102,030,101_78",1_78,1.0,"1,102,030,101",78.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1.0,1277.647095,2376.181152,20151221.0,15.0,43.0
1,"1,102,030,101_86",1_86,1.0,"1,102,030,101",86.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1.0,1277.647095,2376.181152,20151221.0,13.0,35.0
2,"1,102,030,101_26",1_26,1.0,"1,102,030,101",26.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1.0,1277.647095,2376.181152,20151222.0,12.0,4.0
3,"1,102,030,101_172",1_172,1.0,"1,102,030,101",172.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1.0,1277.647095,2376.181152,20151222.0,13.0,23.0
4,"1,102,030,101_164",1_164,1.0,"1,102,030,101",164.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1.0,1277.647095,2376.181152,20151222.0,15.0,8.0
5,"1,102,030,101_146",1_146,1.0,"1,102,030,101",146.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1.0,1277.647095,2376.181152,20151221.0,12.0,51.0
6,"1,102,030,101_43",1_43,1.0,"1,102,030,101",43.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1.0,1277.647095,2376.181152,20151221.0,15.0,31.0
7,"1,102,030,101_189",1_189,1.0,"1,102,030,101",189.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,1.0,1277.647095,2376.181152,20151222.0,12.0,45.0


,RESP_KEY,RESP_KEY_EAHH,id_dhs,psu_id,hh,sex,prov,dist,sect,cel,vill,area,hcluster,child_wght,hh_wght,hdate_vf,q1_hh,q1_mm
648,"3,715,050,601_99",78_99,78.0,"3,715,050,601",99.0,1.0,3.0,7.0,15.0,5.0,6.0,NaN,1.0,921.743530,1666.672852,20151204.0,13.0,55.0
810,"4,414,040,901_98",97_98,97.0,"4,414,040,901",98.0,1.0,4.0,4.0,14.0,4.0,9.0,NaN,1.0,835.431458,1529.014038,20151124.0,11.0,58.0
1548,"3,211,010,601_49",181_49,181.0,"3,211,010,601",49.0,2.0,3.0,2.0,11.0,1.0,6.0,NaN,2.0,1314.073608,2244.710938,20151127.0,15.0,17.0
399,"3,203,030,401_10",50_10,50.0,"3,203,030,401",10.0,1.0,3.0,2.0,3.0,3.0,4.0,NaN,1.0,1158.455688,2094.689697,20151202.0,9.0,44.0
1525,"3,203,040,801_46",179_46,179.0,"3,203,040,801",46.0,2.0,3.0,2.0,3.0,4.0,8.0,NaN,2.0,1333.891113,2278.563232,20151125.0,13.0,0.0
638,"3,710,011,101_96",76_96,76.0,"3,710,011,101",96.0,1.0,3.0,7.0,10.0,1.0,11.0,NaN,1.0,1015.627563,1836.431519,20151126.0,10.0,22.0


### Slot summaries (ID / geo / design)


In [12]:
L = meta.column_names_to_labels or {}

_WORD_SEX = re.compile(r"\b(?:male|females?|female)\b", re.IGNORECASE)
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title: str, cols: list, note_extra: str = ""):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (str(L.get(c) or ""))[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary(
    "1. Respondent row keys (derived)",
    ["RESP_KEY", "RESP_KEY_EAHH"],
    "Equivalent uniqueness: RESP_KEY uses psu_id+hh; RESP_KEY_EAHH uses id_dhs+hh (shorter if linking EA codes)",
)
slot_summary(
    "2. id_dhs alone (EA / DHS frame — not a respondent ID)",
    ["id_dhs"],
    "Best single column for **EA** linkage, **not** next-best for **person**; must add `hh` for unique row",
)
slot_summary("3. Primary sampling unit", ["psu_id"], "**13 chars** including commas (e.g. 1,102,030,101)")
slot_summary("4. Household number", ["hh"], "within PSU; **1–3** integer digits (as string) here")
slot_summary(
    "5. Geography codes",
    ["prov", "dist", "id_dhs", "ea_code", "psu_id", "sect", "cel", "vill"],
    "**GeoLevel2 / Admin 2** = EA (**id_dhs**, **ea_code**, **psu_id**); **sect/cel/vill** = finer (GeoLevel3+)",
)
slot_summary("6. area (all missing in extract)", ["area"], "")
slot_summary("7. hcluster (split-sample indicator)", ["hcluster"], "perfectly aligns with sex in this file (1 with sex 1 only; 2 with sex 2)")
slot_summary("8. Weights", ["child_wght", "hh_wght"], "Do-file uses child_wght with svyset")
slot_summary("9. Final visit date", ["hdate_vf"], "YYYYMMDD as float")
slot_summary("10. Interview start", ["q1_hh", "q1_mm"], "also H1_* head-of-household pair in file")



1. Respondent row keys (derived)
  RESP_KEY | 
    15-17 chars (string); no male/female text (heuristic); dtype=str; n_distinct=2212; missing=0
  RESP_KEY_EAHH | 
    3-7 chars (string); no male/female text (heuristic); dtype=str; n_distinct=2212; missing=0
   Equivalent uniqueness: RESP_KEY uses psu_id+hh; RESP_KEY_EAHH uses id_dhs+hh (shorter if linking EA codes)

2. id_dhs alone (EA / DHS frame — not a respondent ID)
  id_dhs | EA ID
    1-3 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=1.0/250.0; n_distinct=250; missing=0
   Best single column for **EA** linkage, **not** next-best for **person**; must add `hh` for unique row

3. Primary sampling unit
  psu_id | Primary sampling unit ID
    13 chars (string); no male/female text (heuristic); dtype=str; n_distinct=250; missing=0
   **13 chars** including commas (e.g. 1,102,030,101)

4. Household number
  hh | HOUSEHOLD
    1-3 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=1

## 4. Harmonized codebook slots (Rwanda 2015–16)

**Single merged file** with both sexes — **`variable_male`** and **`variable_female`** repeat the **same** Stata names.

**Survey setup (from `VACS_analysis_dofile.do`):** `svyset psu_id [pweight = child_wght], strata(prov)`

### Respondent / row ID — **`id_dhs` is not the next-best alone**

- **`id_dhs`** = **EA ID** (250 EAs / 2,212 rows here). It is the **best single column for DHS / EA-level linkage**, but **wrong as a respondent key** because many adolescents share one EA.
- **Next-best unique row identifiers (equivalent in this file):** **`psu_id` + `hh`** or **`id_dhs` + `hh`**. Here **`psu_id` ↔ `id_dhs` is one-to-one**, so the two composites carry the same information; use **`id_dhs` + `hh`** if you prefer a short numeric EA code for merging.

```
slot	variable_male	variable_female	type_and_width	notes
Respondent row key (PSU string)	(derive) str(psu_id)+'_'+str(int(hh))	same	**psu_id** is **13 chars** with commas	Matches **`svyset`** cluster id + household index; no single-column PUD_ID in file
Respondent row key (EA + HH)	(derive) str(int(id_dhs))+'_'+str(int(hh))	same	shorter composite (**1–3** + **_** + **1–3** digit runs typical)	**Same rows uniquely identified** as line above; better when joining on **EA / DHS** integer codes
EA id (not respondent-unique)	id_dhs	same	float integer **1–250**	**Do not use alone** as respondent ID—**~2–18** rows per EA (median **~9** in this extract)
Cluster (svy)	psu_id	same	**13-char str** (digits + commas)	**Primary sampling unit**; matches **do-file** cluster
Stratum (svy)	prov	same	integer province code (**1–5**)	**`strata(prov)`** in do-file
Household index	hh	same	**1–3 digits** (int string) within PSU	max **18** households per **`psu_id`** here
Geo level 1	prov	same	integer	Province (**Admin 1**)
District	dist	same	integer **1–8**	Within province—layer **above** EA (not **Admin 2** in this project)
Geo level 2 (enumeration area)	id_dhs ea_code	same	**id_dhs** float **1–250**; **ea_code** 8-char str	**Admin 2**: EA identifiers; **1–1** with **psu_id** (see Cluster row)
Geo level 3	sect	same	integer **1–18**	Sector—**finer** than EA (Stata **SECT**)
Finer geo	cel vill	cel vill	integer codes (**cel** 9 levels; **vill** 14)	Cellule / village
Split-sample type	hcluster	same	**1** / **2** numeric	Stata **CLUSTER TYPE**; in this extract **aligns with `sex`** (design, not a geo code)
area	area	area	—	**All missing** in this `.dta`—do not map until confirmed
Sex	sex	same	**1** / **2**	Confirm coding in questionnaire / labels
Analysis weight	child_wght	same	float	**`pweight`** in do-file
Household weight	hh_wght	same	float	Labeled final HH weight; analysis uses **`child_wght`** in cited `svyset`
Final visit date	hdate_vf	same	numeric **YYYYMMDD** (float)	**20151101–20151223** in this extract
Interview start	q1_hh q1_mm	q1_hh q1_mm	integer hour / minute	Respondent interview began; **H1_** pair also present (HoH module)
```

**SAS:** PV/SV/overall `.sas` files in the same folder may define formats for analysis subsets—use if you need label mappings beyond the `.dta`.
